# Task 5 – Backtesting Optimised Portfolios vs Benchmark

**Objective:** Simulate the two optimal portfolios from Task 4 (Max Sharpe and Min Volatility)
over the out-of-sample period **Jan 2025 – Jan 2026** and compare them against a classic
**60 % SPY / 40 % BND** benchmark.

**Metrics computed:**
- Cumulative Returns
- CAGR (Compound Annual Growth Rate)
- Annualised Volatility
- Sharpe Ratio
- Maximum Drawdown

In [1]:
import json
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../data/processed')
IMG_DIR    = Path('../notebooks/images')
IMG_DIR.mkdir(parents=True, exist_ok=True)

STATS_JSON = DATA_DIR / 'task4_stats.json'

# ── Backtest window ──────────────────────────────────────────────────────────
BT_START = '2025-01-01'
BT_END   = '2026-01-15'
RISK_FREE = 0.0425          # same as Task 4

print('Libraries loaded.')
print(f'Backtest window: {BT_START} → {BT_END}')

Libraries loaded.
Backtest window: 2025-01-01 → 2026-01-15


## 1  Load Price Data

In [11]:
def load_asset(ticker: str) -> pd.Series:
    """Return daily Close price series from the saved clean CSV (multi-level header)."""
    df = pd.read_csv(
        DATA_DIR / f'{ticker}_clean.csv',
        header=[0, 1],   # two-row header: Price + Ticker
        index_col=0,
        parse_dates=True,
    )
    close_col = ('Close', ticker)
    series = df[close_col] if close_col in df.columns else df.iloc[:, 0]
    series.index.name = 'Date'
    return series.rename(ticker)

tsla = load_asset('TSLA')
bnd  = load_asset('BND')
spy  = load_asset('SPY')

prices    = pd.concat([tsla, bnd, spy], axis=1).sort_index()
bt_prices = prices.loc[BT_START:BT_END].copy()
bt_prices.dropna(inplace=True)

print(f'Backtest rows: {len(bt_prices)}')
display(bt_prices.head())

Backtest rows: 259


,TSLA,BND,SPY
Date,,,
2025-01-02,379.279999,68.971504,577.854126
2025-01-03,410.440002,68.885216,585.079285
2025-01-06,411.049988,68.818100,588.449768
2025-01-07,394.359985,68.578415,581.797913
2025-01-08,394.940002,68.655128,582.647888


## 2  Define Portfolio Weights

In [13]:
with open(STATS_JSON) as f:
    t4 = json.load(f)

w_max_sharpe = t4['max_sharpe']['weights']
w_min_vol    = t4['min_volatility']['weights']
w_benchmark  = {'TSLA': 0.0, 'BND': 0.40, 'SPY': 0.60}

TICKERS = ['TSLA', 'BND', 'SPY']

portfolios = {
    'Max Sharpe':      np.array([w_max_sharpe[t] for t in TICKERS]),
    'Min Volatility':  np.array([w_min_vol[t]    for t in TICKERS]),
    'Benchmark 60/40': np.array([w_benchmark[t]  for t in TICKERS]),
}

print('Portfolio weights:')
display(pd.DataFrame(portfolios, index=TICKERS).T)

Portfolio weights:


,TSLA,BND,SPY
Max Sharpe,0.0,0.00000,1.00000
Min Volatility,0.0,0.92903,0.07097
Benchmark 60/40,0.0,0.40000,0.60000


## 3  Compute Daily Returns & Backtest

In [14]:
daily_returns = bt_prices.pct_change().dropna()

cumulative: dict[str, pd.Series] = {}
for name, weights in portfolios.items():
    port_ret = daily_returns[TICKERS].dot(weights)
    cumulative[name] = (1 + port_ret).cumprod()

cum_df = pd.DataFrame(cumulative)
print('Cumulative returns (first 5 rows):')
display(cum_df.head())

Cumulative returns (first 5 rows):


,Max Sharpe,Min Volatility,Benchmark 60/40
Date,,,
2025-01-03,1.012503,0.999725,1.007002
2025-01-06,1.018336,0.999229,1.010090
2025-01-07,1.006825,0.995194,1.001832
2025-01-08,1.008296,0.996331,1.003158
2025-01-10,0.992902,0.990469,0.991895


## 4  Performance Metrics

In [15]:
TRADING_DAYS = 252

def compute_metrics(port_returns: pd.Series, risk_free: float = RISK_FREE) -> dict:
    n_days  = len(port_returns)
    total_r = (1 + port_returns).prod() - 1
    years   = n_days / TRADING_DAYS
    cagr    = (1 + total_r) ** (1 / years) - 1 if years > 0 else float('nan')

    ann_vol = port_returns.std() * np.sqrt(TRADING_DAYS)
    ann_ret = port_returns.mean() * TRADING_DAYS
    sharpe  = (ann_ret - risk_free) / ann_vol if ann_vol != 0 else float('nan')

    cum     = (1 + port_returns).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()

    return {
        'Total Return':    round(total_r * 100, 2),
        'CAGR':            round(cagr    * 100, 2),
        'Ann. Volatility': round(ann_vol * 100, 2),
        'Sharpe Ratio':    round(sharpe,         4),
        'Max Drawdown':    round(max_dd  * 100, 2),
    }

metrics_rows = {}
for name, weights in portfolios.items():
    port_ret = daily_returns[TICKERS].dot(weights)
    metrics_rows[name] = compute_metrics(port_ret)

metrics_df = pd.DataFrame(metrics_rows).T
metrics_df.index.name = 'Portfolio'
print('Performance metrics:')
display(metrics_df)

Performance metrics:


,Total Return,CAGR,Ann. Volatility,Sharpe Ratio,Max Drawdown
Portfolio,,,,,
Max Sharpe,19.47,18.98,19.22,0.7780,-18.76
Min Volatility,8.50,8.29,4.43,0.8619,-2.26
Benchmark 60/40,15.07,14.70,11.82,0.8594,-11.29


## 5  Visualisations

In [16]:
colours = {'Max Sharpe': '#2196F3', 'Min Volatility': '#4CAF50', 'Benchmark 60/40': '#FF9800'}
styles  = {'Max Sharpe': '-',       'Min Volatility': '--',      'Benchmark 60/40': ':'}

fig, ax = plt.subplots(figsize=(12, 5))
for name, series in cum_df.items():
    ax.plot(series.index, series.values, label=name,
            color=colours[name], linestyle=styles[name], lw=2)

ax.axhline(1.0, color='grey', lw=0.8, linestyle='--', alpha=0.5)
ax.set_title('Cumulative Returns – Backtesting Jan 2025 – Jan 2026', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Portfolio Value ($1 invested)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / 't5_fig1_cumulative_returns.png', dpi=150)
plt.show()
print('Saved: t5_fig1_cumulative_returns.png')

Saved: t5_fig1_cumulative_returns.png


In [17]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, weights in portfolios.items():
    pr  = daily_returns[TICKERS].dot(weights)
    cum = (1 + pr).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax() * 100
    ax.fill_between(dd.index, dd.values, alpha=0.3, color=colours[name], label=name)
    ax.plot(dd.index, dd.values, color=colours[name], lw=1.2)

ax.set_title('Portfolio Drawdown – Backtesting Period', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Drawdown (%)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / 't5_fig2_drawdown.png', dpi=150)
plt.show()
print('Saved: t5_fig2_drawdown.png')

Saved: t5_fig2_drawdown.png


In [18]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
bar_metrics = ['Total Return', 'CAGR', 'Sharpe Ratio', 'Max Drawdown']
bar_colors  = [colours[n] for n in metrics_df.index]

for ax, metric in zip(axes, bar_metrics):
    vals = metrics_df[metric]
    bars = ax.bar(metrics_df.index, vals, color=bar_colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.set_xticks(range(len(metrics_df)))
    ax.set_xticklabels(metrics_df.index, rotation=20, ha='right', fontsize=8)
    ax.axhline(0, color='grey', lw=0.8); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Performance Metrics Comparison', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(IMG_DIR / 't5_fig3_metrics_comparison.png', dpi=150)
plt.show()
print('Saved: t5_fig3_metrics_comparison.png')

Saved: t5_fig3_metrics_comparison.png


## 6  Save Backtesting Stats

In [19]:
task5_stats = {
    'backtest_period': {'start': BT_START, 'end': BT_END},
    'risk_free_rate': RISK_FREE,
    'portfolios': {
        name: {
            'weights': dict(zip(TICKERS, w.tolist())),
            'metrics': metrics_rows[name],
        }
        for name, w in portfolios.items()
    },
}

out_path = DATA_DIR / 'task5_stats.json'
with open(out_path, 'w') as f:
    json.dump(task5_stats, f, indent=2)

print(f'Stats saved → {out_path}')
print(json.dumps(task5_stats, indent=2))

Stats saved → ../data/processed/task5_stats.json
{
  "backtest_period": {
    "start": "2025-01-01",
    "end": "2026-01-15"
  },
  "risk_free_rate": 0.0425,
  "portfolios": {
    "Max Sharpe": {
      "weights": {
        "TSLA": 0.0,
        "BND": 0.0,
        "SPY": 1.0
      },
      "metrics": {
        "Total Return": 19.47,
        "CAGR": 18.98,
        "Ann. Volatility": 19.22,
        "Sharpe Ratio": 0.778,
        "Max Drawdown": -18.76
      }
    },
    "Min Volatility": {
      "weights": {
        "TSLA": 0.0,
        "BND": 0.92903,
        "SPY": 0.07097
      },
      "metrics": {
        "Total Return": 8.5,
        "CAGR": 8.29,
        "Ann. Volatility": 4.43,
        "Sharpe Ratio": 0.8619,
        "Max Drawdown": -2.26
      }
    },
    "Benchmark 60/40": {
      "weights": {
        "TSLA": 0.0,
        "BND": 0.4,
        "SPY": 0.6
      },
      "metrics": {
        "Total Return": 15.07,
        "CAGR": 14.7,
        "Ann. Volatility": 11.82,
        "Shar

## 7  Conclusion

### Key Findings

The backtesting experiment compared two theoretically optimal portfolios from Task 4 against a
standard 60/40 benchmark over the Jan 2025 – Jan 2026 period.

#### Max Sharpe Portfolio (100 % SPY)
- Because the TSLA expected return was **negative** (−38.7 %), PyPortfolioOpt pushed the
  entire allocation into SPY, producing the same performance as a pure SPY position.
- This portfolio captured the strong equity rally of early-to-mid 2025 but was also fully
  exposed to its drawdowns.

#### Min Volatility Portfolio (~93 % BND, ~7 % SPY)
- Heavy bond weighting minimises short-term variance but sacrifices upside capture.
- Delivered a near-flat, low-volatility trajectory — appropriate for risk-averse mandates.

#### Benchmark (60 % SPY / 40 % BND)
- Provided a balanced trade-off: moderate return with constrained drawdown.
- In years when equities outperform, the 60/40 mix under-performs pure SPY but significantly
  outperforms heavy-bond portfolios.

### Limitations & Next Steps
1. **Negative TSLA expected return** (from LSTM forecast) drives zero TSLA allocation;
   ensemble averaging or longer training windows might produce more stable forecasts.
2. **Single-period optimisation** – MVO weights are static; a dynamic re-balancing strategy
   (e.g., monthly) could reduce tracking error.
3. **Transaction costs** are not modelled; real-world slippage would favour less-frequent
   rebalancing.
4. **Out-of-sample coverage is only one year** – longer horizons would yield more statistically
   reliable conclusions.